# CampusHelp: Student FAQ Semantic Search

This notebook loads FAQ data, cleans it, stores the FAQ text in Chroma, and searches for answers to a question.





## 1. Load the FAQ data


In [5]:
from IPython.display import display
import pandas as pd

df = pd.read_csv("faqs.csv")
print("Number of FAQs:", len(df))
display(df.head())

Number of FAQs: 10


,id,category,question,answer
0,1,Registration,How do I register for courses?,Log in to the student portal and follow your d...
1,2,Registration,What if I miss the course registration deadline?,Contact your department or registrar to ask wh...
2,3,Library,When can I use the university library?,Check the library's published opening hours fo...
3,4,Library,Can I find a quiet place to study?,The university library may provide quiet study...
4,5,Exams,How can I check my exam results?,Check the student portal or follow the officia...


## 2. Clean the data


In [ ]:
# Remove rows that do not have a question or answer.
df = df.dropna(subset=["question", "answer"])

# Remove extra spaces and empty entries.
for col in ["category", "question", "answer"]:
    df[col] = df[col].astype(str).str.strip()

df = df[(df["question"] != "") & (df["answer"] != "")]
df = df.drop_duplicates(subset=["question"]).reset_index(drop=True)

print("FAQs after cleaning:", len(df))
display(df.head())

FAQs after cleaning: 10


,id,category,question,answer
0,1,Registration,How do I register for courses?,Log in to the student portal and follow your d...
1,2,Registration,What if I miss the course registration deadline?,Contact your department or registrar to ask wh...
2,3,Library,When can I use the university library?,Check the library's published opening hours fo...
3,4,Library,Can I find a quiet place to study?,The university library may provide quiet study...
4,5,Exams,How can I check my exam results?,Check the student portal or follow the officia...


## 3. Make text chunks

Each FAQ is short, so I use one chunk for each FAQ. This keeps the question and its answer together.


In [10]:
chunks = []

for _, row in df.iterrows():
    text = (
        "Category: " + row["category"] + "\n" +
        "Question: " + row["question"] + "\n" +
        "Answer: " + row["answer"]
    )
    chunks.append({
        "id": str(row["id"]),
        "category": row["category"],
        "question": row["question"],
        "text": text
    })

print("Chunks created:", len(chunks))
print(chunks[0]["text"])

Chunks created: 10
Category: Registration
Question: How do I register for courses?
Answer: Log in to the student portal and follow your department's registration instructions.


## 4. Store the chunks in Chroma

Chroma uses an embedding model to turn text into vectors. On the first run, its default model may need to download files, so this step needs an internet connection if the model is not already installed.


In [11]:
import chromadb
from chromadb.utils import embedding_functions

client = chromadb.PersistentClient(path="./chroma_db")
embedding_function = embedding_functions.DefaultEmbeddingFunction()
# Recreate the collection so old test data does not mix with this run.
try:
    client.delete_collection("campus_faqs")
except Exception:
    pass

collection = client.create_collection(
    name="campus_faqs",
    embedding_function=embedding_function
)
documents = [item["text"] for item in chunks]
ids = ["faq_" + item["id"] for item in chunks]
metadatas = [
    {"category": item["category"], "question": item["question"]}
    for item in chunks
]

collection.add(ids=ids, documents=documents, metadatas=metadatas)
print("FAQs stored in Chroma:", collection.count())

C:\Users\hp\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz:   0%|          | 305k/79.3M [00:22<1:40:25, 13.8kiB/s] 


ReadTimeout: The read operation timed out in add.

## 5. Search for a question

Try changing the question below. Chroma compares its embedding with the stored FAQ embeddings and returns the closest matches.


In [ ]:
def search_faq(question, number_of_results=3):
    results = collection.query(
        query_texts=[question],
        n_results=min(number_of_results, collection.count())
    )

    for i, text in enumerate(results["documents"][0]):
        print("Result", i + 1)
        print("Question:", results["metadatas"][0][i]["question"])
        print("Category:", results["metadatas"][0][i]["category"])
        print("FAQ text:", text)
        print("-" * 40)

search_faq("How can I change my student portal password?")

## 6. Try other queries


In [ ]:
search_faq("I missed the deadline to choose my courses")
search_faq("Where do I see my exam marks?")
search_faq("Where can I study quietly?")

NameError: name 'search_faq' is not defined